In [1]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType
)
from pyspark.sql.window import Window
import pandas as pd

# Synapse notebooks already provide a `spark` session.
# This line is a safe no-op in Synapse and allows local/testing execution too.
spark = SparkSession.builder.appName("RetailMedallionPipeline").getOrCreate()

spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")


StatementMeta(kmspark, 1, 2, Finished, Available, Finished, False)

In [3]:
# ---- ADLS Gen2 connection details ----
storage_account_name = "knoledgers"
container_name        = "retail"

# Base ADLS Gen2 path (abfss). Synapse workspaces with a linked ADLS Gen2 account
# can use this scheme directly without extra auth config.
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"

# ---- Folder structure ----
landing_path = f"{base_path}/raw/raw retail"          # raw incoming daily files (csv/xlsx)
bronze_path  = f"{base_path}/bronze/transactions"
silver_path  = f"{base_path}/silver/transactions"
gold_path    = f"{base_path}/gold"

# ---- Daily batch identifier ----
# In production this would come from a pipeline parameter (e.g. Synapse Pipeline trigger time).
batch_date = "2026-07-07"

# ---- Source file(s) for today's batch ----
# Add as many files as land for the day; both .csv and .xlsx are supported.
source_files = [
    f"{landing_path}/Retail_Transactions_Dataset.csv",
    # f"{landing_path}/{batch_date}/transactions_{batch_date}.xlsx",
]

print("Bronze path :", bronze_path)
print("Silver path :", silver_path)
print("Gold path   :", gold_path)


StatementMeta(kmspark, 2, 4, Finished, Available, Finished, False)

Bronze path : abfss://retail@knoledgers.dfs.core.windows.net/bronze/transactions
Silver path : abfss://retail@knoledgers.dfs.core.windows.net/silver/transactions
Gold path   : abfss://retail@knoledgers.dfs.core.windows.net/gold


In [3]:
# Explicit raw schema — everything as string, casting happens later in Silver
raw_schema = StructType([
    StructField("Transaction_ID",     StringType(), True),
    StructField("Date",               StringType(), True),
    StructField("Customer_Name",      StringType(), True),
    StructField("Product",            StringType(), True),
    StructField("Total_Items",        StringType(), True),
    StructField("Total_Cost",         StringType(), True),
    StructField("Payment_Method",     StringType(), True),
    StructField("City",               StringType(), True),
    StructField("Store_Type",         StringType(), True),
    StructField("Discount_Applied",   StringType(), True),
    StructField("Customer_Category",  StringType(), True),
    StructField("Season",             StringType(), True),
    StructField("Promotion",          StringType(), True),
])


def read_csv_file(path: str) -> DataFrame:
    """Read a single CSV file into a DataFrame using the fixed raw schema."""
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .schema(raw_schema)
        .csv(path)
    )


def read_excel_file(path: str) -> DataFrame:
    """
    Read a single Excel file into a Spark DataFrame.

    Spark has no native Excel reader, so we read the file with pandas (via the driver)
    and convert it to a Spark DataFrame. This is fine for daily batch files of a
    reasonable size; for very large Excel files, convert to CSV upstream instead.
    """
    pdf = pd.read_excel(path, dtype=str)  # read everything as string, same as raw_schema
    pdf = pdf.reindex(columns=[f.name for f in raw_schema.fields])  # align column order
    return spark.createDataFrame(pdf, schema=raw_schema)


def read_any(path: str) -> DataFrame:
    """Dispatch to the correct reader based on file extension."""
    if path.lower().endswith(".csv"):
        return read_csv_file(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        return read_excel_file(path)
    else:
        raise ValueError(f"Unsupported file format for: {path}")


def load_daily_batch(paths: list) -> DataFrame:
    """Read and union all files (csv and/or xlsx) that landed for the current batch."""
    dfs = [read_any(p) for p in paths]
    df = dfs[0]
    for other in dfs[1:]:
        df = df.unionByName(other)
    return df


raw_df = load_daily_batch(source_files)
print(f"Loaded {raw_df.count()} raw rows from {len(source_files)} file(s)")
raw_df.show(5, truncate=False)


StatementMeta(kmspark, 1, 4, Finished, Available, Finished, False)

Loaded 1000000 raw rows from 1 file(s)
+--------------+-------------------+-----------------+-------------------------------------------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------------+
|Transaction_ID|Date               |Customer_Name    |Product                                                |Total_Items|Total_Cost|Payment_Method|City         |Store_Type      |Discount_Applied|Customer_Category|Season|Promotion                 |
+--------------+-------------------+-----------------+-------------------------------------------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------------+
|1000000000    |2022-01-21 06:27:29|Stacey Price     |['Ketchup', 'Shaving Cream', 'Light Bulbs']            |3          |71.65     |Mobile Payment|Los Angeles  |Warehouse Club  |True            |Homemaker        |

In [4]:
bronze_df = (
    raw_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_batch_date", F.lit(batch_date))
)

print("Bronze schema:")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)


StatementMeta(kmspark, 1, 5, Finished, Available, Finished, False)

Bronze schema:
root
 |-- Transaction_ID: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: string (nullable = true)
 |-- Total_Cost: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: string (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_batch_date: string (nullable = false)

+--------------+-------------------+-----------------+-------------------------------------------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------------+--------------------------+-----------------+
|Transaction_ID|Date         

In [21]:
text_columns = [
    "Customer_Name", "Payment_Method", "City", "Store_Type",
    "Customer_Category", "Season", "Promotion"
]

silver_df = bronze_df

# Trim whitespace on every string column, Bronze's audit columns included
for col_name in bronze_df.columns:
    if dict(bronze_df.dtypes)[col_name] == "string":
        silver_df = silver_df.withColumn(col_name, F.trim(F.col(col_name)))

# Standardize categorical/text columns to a consistent case
# (Title Case for names/cities, as it reads best for reporting)
for col_name in ["Customer_Name", "City", "Store_Type", "Customer_Category", "Season", "Promotion", "Payment_Method"]:
    silver_df = silver_df.withColumn(col_name, F.initcap(F.col(col_name)))

# Normalize obviously malformed / placeholder values to real nulls
malformed_tokens = ["", "NA", "N/A", "NULL", "None", "-"]
for col_name in text_columns:
    silver_df = silver_df.withColumn(
        col_name,
        F.when(F.col(col_name).isin(malformed_tokens), None).otherwise(F.col(col_name))
    )


StatementMeta(kmspark, 1, 22, Finished, Available, Finished, False)

In [6]:
silver_df = (
    silver_df
    # Date -> timestamp. Adjust the format string to match the real source format.
    .withColumn("Date", F.to_timestamp(F.col("Date"), "yyyy-MM-dd"))
    # Total_Items -> integer
    .withColumn("Total_Items", F.col("Total_Items").cast(IntegerType()))
    # Total_Cost -> double
    .withColumn("Total_Cost", F.col("Total_Cost").cast(DoubleType()))
    # Discount_Applied -> boolean (handles Yes/No, True/False, 1/0 style source values)
    .withColumn(
        "Discount_Applied",
        F.when(F.lower(F.col("Discount_Applied")).isin("yes", "true", "1"), True)
         .when(F.lower(F.col("Discount_Applied")).isin("no", "false", "0"), False)
         .otherwise(None)
         .cast(BooleanType())
    )
)


StatementMeta(kmspark, 1, 7, Finished, Available, Finished, False)

In [23]:
silver_df = (
    silver_df
    .withColumn("Year", F.year("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("Day", F.dayofmonth("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("Month_Name", F.date_format("Date", "MMMM"))
)


StatementMeta(kmspark, 1, 24, Finished, Available, Finished, False)

In [24]:
silver_df = silver_df.withColumn(
    "Product_Array",
    F.transform(F.split(F.col("Product"), ","), lambda x: F.trim(x))
)


StatementMeta(kmspark, 1, 25, Finished, Available, Finished, False)

In [25]:
silver_df_validated = silver_df.filter(
    F.col("Transaction_ID").isNotNull()
    & F.col("Date").isNotNull()
    & F.col("Total_Items").isNotNull() & (F.col("Total_Items") > 0)
    & F.col("Total_Cost").isNotNull() & (F.col("Total_Cost") >= 0)
    & (F.size(F.col("Product_Array")) > 0)
)

dropped = silver_df.count() - silver_df_validated.count()
print(f"Silver validation dropped {dropped} invalid row(s)")

silver_df_final = silver_df_validated.dropDuplicates(["Transaction_ID"])

print("Silver schema:")
silver_df_final.printSchema()
silver_df_final.show(5, truncate=False)


StatementMeta(kmspark, 1, 26, Finished, Available, Finished, False)

Silver validation dropped 1000000 invalid row(s)
Silver schema:
root
 |-- Transaction_ID: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: boolean (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_batch_date: string (nullable = false)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month_Name: string (nullable = true)
 |-- Product_Array: array (nullable = true)
 |    |-- element: string (contai

In [26]:
customer_window = Window.orderBy("Customer_Name", "Customer_Category")

dim_customer = (
    silver_df_final
    .select("Customer_Name", "Customer_Category")
    .distinct()
    .withColumn("Customer_Key", F.row_number().over(customer_window))
    .select("Customer_Key", "Customer_Name", "Customer_Category")
)

dim_customer.show(5, truncate=False)


StatementMeta(kmspark, 1, 27, Finished, Available, Finished, False)

+------------+-------------+-----------------+
|Customer_Key|Customer_Name|Customer_Category|
+------------+-------------+-----------------+
+------------+-------------+-----------------+



In [27]:
product_window = Window.orderBy("Product_Name")

dim_product = (
    silver_df_final
    .select(F.explode("Product_Array").alias("Product_Name"))
    .distinct()
    .withColumn("Product_Key", F.row_number().over(product_window))
    .select("Product_Key", "Product_Name")
)

dim_product.show(5, truncate=False)


StatementMeta(kmspark, 1, 28, Finished, Available, Finished, False)

+-----------+------------+
|Product_Key|Product_Name|
+-----------+------------+
+-----------+------------+



In [29]:
date_window = Window.orderBy("Full_Date")

dim_date = (
    silver_df_final
    .select(
        F.col("Date").alias("Full_Date"),
        "Day", "Month", "Month_Name", "Quarter", "Year", "Season"
    )
    .distinct()
    .withColumn("Date_Key", F.row_number().over(date_window))
    .select("Date_Key", "Full_Date", "Day", "Month", "Month_Name", "Quarter", "Year", "Season")
)

dim_date.show(5, truncate=False)


StatementMeta(kmspark, 1, 30, Finished, Available, Finished, False)

+--------+---------+---+-----+----------+-------+----+------+
|Date_Key|Full_Date|Day|Month|Month_Name|Quarter|Year|Season|
+--------+---------+---+-----+----------+-------+----+------+
+--------+---------+---+-----+----------+-------+----+------+



In [30]:
store_window = Window.orderBy("City", "Store_Type")

dim_store = (
    silver_df_final
    .select("City", "Store_Type")
    .distinct()
    .withColumn("Store_Key", F.row_number().over(store_window))
    .select("Store_Key", "City", "Store_Type")
)

dim_store.show(5, truncate=False)


StatementMeta(kmspark, 1, 31, Finished, Available, Finished, False)

+---------+----+----------+
|Store_Key|City|Store_Type|
+---------+----+----------+
+---------+----+----------+



In [31]:
promotion_window = Window.orderBy("Promotion", "Discount_Applied")

dim_promotion = (
    silver_df_final
    .select("Promotion", "Discount_Applied")
    .distinct()
    .withColumn("Promotion_Key", F.row_number().over(promotion_window))
    .select("Promotion_Key", "Promotion", "Discount_Applied")
)

dim_promotion.show(5, truncate=False)


StatementMeta(kmspark, 1, 32, Finished, Available, Finished, False)

+-------------+---------+----------------+
|Promotion_Key|Promotion|Discount_Applied|
+-------------+---------+----------------+
+-------------+---------+----------------+



In [32]:
fact_sales = (
    silver_df_final.alias("s")
    .join(dim_customer.alias("c"), on=["Customer_Name", "Customer_Category"], how="left")
    .join(dim_store.alias("st"), on=["City", "Store_Type"], how="left")
    .join(dim_promotion.alias("p"), on=["Promotion", "Discount_Applied"], how="left")
    .join(dim_date.alias("d"), F.col("s.Date") == F.col("d.Full_Date"), how="left")
    .select(
        F.col("s.Transaction_ID"),
        F.col("d.Date_Key"),
        F.col("c.Customer_Key"),
        F.col("st.Store_Key"),
        F.col("p.Promotion_Key"),
        F.col("s.Total_Items"),
        F.col("s.Total_Cost"),
    )
)

fact_sales.show(5, truncate=False)
fact_sales = (
    silver_df_final.alias("s")
    .join(dim_customer.alias("c"), on=["Customer_Name", "Customer_Category"], how="left")
    .join(dim_store.alias("st"), on=["City", "Store_Type"], how="left")
    .join(dim_promotion.alias("p"), on=["Promotion", "Discount_Applied"], how="left")
    .join(dim_date.alias("d"), F.col("s.Date") == F.col("d.Full_Date"), how="left")
    .select(
        F.col("s.Transaction_ID"),
        F.col("d.Date_Key"),
        F.col("c.Customer_Key"),
        F.col("st.Store_Key"),
        F.col("p.Promotion_Key"),
        F.col("s.Total_Items"),
        F.col("s.Total_Cost"),
    )
)

fact_sales.show(5, truncate=False)


StatementMeta(kmspark, 1, 33, Finished, Available, Finished, False)

+--------------+--------+------------+---------+-------------+-----------+----------+
|Transaction_ID|Date_Key|Customer_Key|Store_Key|Promotion_Key|Total_Items|Total_Cost|
+--------------+--------+------------+---------+-------------+-----------+----------+
+--------------+--------+------------+---------+-------------+-----------+----------+

+--------------+--------+------------+---------+-------------+-----------+----------+
|Transaction_ID|Date_Key|Customer_Key|Store_Key|Promotion_Key|Total_Items|Total_Cost|
+--------------+--------+------------+---------+-------------+-----------+----------+
+--------------+--------+------------+---------+-------------+-----------+----------+



In [33]:
fact_product_bridge = (
    silver_df_final
    .select("Transaction_ID", F.explode("Product_Array").alias("Product_Name"))
    .join(dim_product, on="Product_Name", how="left")
    .select("Transaction_ID", "Product_Key")
)

fact_product_bridge.show(5, truncate=False)


StatementMeta(kmspark, 1, 34, Finished, Available, Finished, False)

+--------------+-----------+
|Transaction_ID|Product_Key|
+--------------+-----------+
+--------------+-----------+



In [35]:
def write_csv(df: DataFrame, path: str, partition_col: str = None):
    """Write a DataFrame as CSV with header, overwriting any existing data at the path."""
    writer = df.coalesce(1).write.mode("overwrite").option("header", True)
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.csv(path)


# ---- Bronze ----
write_csv(bronze_df, f"{bronze_path}/batch_date={batch_date}")

# ---- Silver ----
# نحول Product_Array لنص مفصول بفاصلة عشان CSV يقدر يستوعبه
silver_df_to_write = silver_df_final.withColumn(
    "Product_Array", F.concat_ws(", ", F.col("Product_Array"))
)
write_csv(silver_df_to_write, silver_path)

# ---- Gold: dimensions + fact + bridge ----
write_csv(dim_customer,        f"{gold_path}/DimCustomer")
write_csv(dim_product,         f"{gold_path}/DimProduct")
write_csv(dim_date,            f"{gold_path}/DimDate")
write_csv(dim_store,           f"{gold_path}/DimStore")
write_csv(dim_promotion,       f"{gold_path}/DimPromotion")
write_csv(fact_sales,          f"{gold_path}/FactSales")
write_csv(fact_product_bridge, f"{gold_path}/FactProductBridge")

print("Bronze, Silver, and Gold layers written successfully.")
print(f"Bronze -> {bronze_path}/batch_date={batch_date}")
print(f"Silver -> {silver_path}")
print(f"Gold   -> {gold_path}")

StatementMeta(kmspark, 1, 36, Finished, Available, Finished, False)

Bronze, Silver, and Gold layers written successfully.
Bronze -> abfss://retail@knoledgers.dfs.core.windows.net/bronze/transactions/batch_date=2026-07-07
Silver -> abfss://retail@knoledgers.dfs.core.windows.net/silver/transactions
Gold   -> abfss://retail@knoledgers.dfs.core.windows.net/gold


In [ ]:
def write_csv(df: DataFrame, path: str, partition_col: str = None):
    """Write a DataFrame as CSV with header, overwriting any existing data at the path."""
    writer = df.coalesce(1).write.mode("overwrite").option("header", True)
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.csv(path)


# ---- Bronze ----
write_csv(bronze_df, f"{bronze_path}/batch_date={batch_date}")

# ---- Silver ----
# نحول Product_Array لنص مفصول بفاصلة عشان CSV يقدر يستوعبه
# (بنستخدم نسخة منفصلة عشان silver_df_final الأصلي يفضل زي ما هو للاستخدام في Gold)
silver_df_to_write = silver_df_final.withColumn(
    "Product_Array", F.concat_ws(", ", F.col("Product_Array"))
)
write_csv(silver_df_to_write, silver_path)

# ---- Gold: dimensions + fact + bridge ----
write_csv(dim_customer,        f"{gold_path}/DimCustomer")
write_csv(dim_product,         f"{gold_path}/DimProduct")
write_csv(dim_date,            f"{gold_path}/DimDate")
write_csv(dim_store,           f"{gold_path}/DimStore")
write_csv(dim_promotion,       f"{gold_path}/DimPromotion")
write_csv(fact_sales,          f"{gold_path}/FactSales")
write_csv(fact_product_bridge, f"{gold_path}/FactProductBridge")

print("Bronze, Silver, and Gold layers written successfully.")
print(f"Bronze -> {bronze_path}/batch_date={batch_date}")
print(f"Silver -> {silver_path}")
print(f"Gold   -> {gold_path}")

StatementMeta(, 1, -1, Cancelled, , Cancelled, True)

In [36]:
print("=== Row Counts ===")
print(f"Bronze         : {bronze_df.count()}")
print(f"Silver         : {silver_df_final.count()}")
print(f"DimCustomer    : {dim_customer.count()}")
print(f"DimProduct     : {dim_product.count()}")
print(f"DimDate        : {dim_date.count()}")
print(f"DimStore       : {dim_store.count()}")
print(f"DimPromotion   : {dim_promotion.count()}")
print(f"FactSales      : {fact_sales.count()}")
print(f"FactProductBridge : {fact_product_bridge.count()}")


StatementMeta(kmspark, 1, 37, Finished, Available, Finished, False)

=== Row Counts ===
Bronze         : 1000000
Silver         : 0
DimCustomer    : 0
DimProduct     : 0
DimDate        : 0
DimStore       : 0
DimPromotion   : 0
FactSales      : 0
FactProductBridge : 0


In [20]:
# 1) عرض أول 5 صفوف من البيانات الخام (Bronze)
print("=== أول 5 صفوف من Bronze ===")
bronze_df.show(5, truncate=False)

# 2) عرض الـ Schema (نوع كل عمود)
print("=== Schema بتاع Bronze ===")
bronze_df.printSchema()

# 3) تشخيص: شوف شكل التاريخ الخام قبل التحويل
print("=== عينة من عمود Date الخام ===")
bronze_df.select("Date").show(5, truncate=False)

# 4) شوف بعد التحويل بقى null ولا لأ
from pyspark.sql import functions as F
test_date = bronze_df.withColumn("Date_parsed", F.to_timestamp(F.col("Date"), "yyyy-MM-dd"))
test_date.select("Date", "Date_parsed").show(5, truncate=False)

# 5) لو عايز تعرف كام صف null بعد كل شرط فلترة على حدة (تشخيص شامل)
print("Nulls after Date parsing:", test_date.filter(F.col("Date_parsed").isNull()).count())

StatementMeta(kmspark, 1, 21, Finished, Available, Finished, False)

=== أول 5 صفوف من Bronze ===
+--------------+-------------------+-----------------+-------------------------------------------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------------+--------------------------+-----------------+
|Transaction_ID|Date               |Customer_Name    |Product                                                |Total_Items|Total_Cost|Payment_Method|City         |Store_Type      |Discount_Applied|Customer_Category|Season|Promotion                 |ingestion_timestamp       |source_batch_date|
+--------------+-------------------+-----------------+-------------------------------------------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------------+--------------------------+-----------------+
|1000000000    |2022-01-21 06:27:29|Stacey Price     |['Ketchup', 'Shaving Cream', 'Light

In [22]:
silver_df = (
    silver_df
    # Date -> timestamp: نجرب أكتر من صيغة شائعة لحد ما نلاقي اللي بيطابق البيانات الحقيقية
    .withColumn(
        "Date",
        F.coalesce(
            F.to_timestamp(F.col("Date"), "yyyy-MM-dd"),
            F.to_timestamp(F.col("Date"), "dd-MM-yyyy"),
            F.to_timestamp(F.col("Date"), "MM-dd-yyyy"),
            F.to_timestamp(F.col("Date"), "dd/MM/yyyy"),
            F.to_timestamp(F.col("Date"), "MM/dd/yyyy"),
            F.to_timestamp(F.col("Date"), "yyyy/MM/dd"),
        )
    )
    # Total_Items -> integer
    .withColumn("Total_Items", F.col("Total_Items").cast(IntegerType()))
    # Total_Cost -> double
    .withColumn("Total_Cost", F.col("Total_Cost").cast(DoubleType()))
    # Discount_Applied -> boolean (handles Yes/No, True/False, 1/0 style source values)
    .withColumn(
        "Discount_Applied",
        F.when(F.lower(F.col("Discount_Applied")).isin("yes", "true", "1"), True)
         .when(F.lower(F.col("Discount_Applied")).isin("no", "false", "0"), False)
         .otherwise(None)
         .cast(BooleanType())
    )
)

StatementMeta(kmspark, 1, 23, Finished, Available, Finished, False)

In [37]:
# نشوف شكل عمود Date الخام زي ما هو من غير أي تحويل
bronze_df.select("Date").show(10, truncate=False)

StatementMeta(kmspark, 1, 38, Finished, Available, Finished, False)

+-------------------+
|Date               |
+-------------------+
|2022-01-21 06:27:29|
|2023-03-01 13:01:21|
|2024-03-21 15:37:04|
|2020-10-31 09:59:47|
|2020-12-10 00:59:59|
|2021-10-07 12:37:26|
|2023-01-08 10:40:03|
|2020-09-03 12:39:59|
|2021-04-05 06:32:18|
|2021-07-08 10:08:59|
+-------------------+
only showing top 10 rows



In [2]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType
)
from pyspark.sql.window import Window
import pandas as pd

# Synapse notebooks already provide a `spark` session.
# This line is a safe no-op in Synapse and allows local/testing execution too.
spark = SparkSession.builder.appName("RetailMedallionPipeline").getOrCreate()

spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")


StatementMeta(kmspark, 2, 3, Finished, Available, Finished, False)

In [4]:
# Explicit raw schema — everything as string, casting happens later in Silver
raw_schema = StructType([
    StructField("Transaction_ID",     StringType(), True),
    StructField("Date",               StringType(), True),
    StructField("Customer_Name",      StringType(), True),
    StructField("Product",            StringType(), True),
    StructField("Total_Items",        StringType(), True),
    StructField("Total_Cost",         StringType(), True),
    StructField("Payment_Method",     StringType(), True),
    StructField("City",               StringType(), True),
    StructField("Store_Type",         StringType(), True),
    StructField("Discount_Applied",   StringType(), True),
    StructField("Customer_Category",  StringType(), True),
    StructField("Season",             StringType(), True),
    StructField("Promotion",          StringType(), True),
])


def read_csv_file(path: str) -> DataFrame:
    """Read a single CSV file into a DataFrame using the fixed raw schema."""
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .schema(raw_schema)
        .csv(path)
    )


def read_excel_file(path: str) -> DataFrame:
    """
    Read a single Excel file into a Spark DataFrame.

    Spark has no native Excel reader, so we read the file with pandas (via the driver)
    and convert it to a Spark DataFrame. This is fine for daily batch files of a
    reasonable size; for very large Excel files, convert to CSV upstream instead.
    """
    pdf = pd.read_excel(path, dtype=str)  # read everything as string, same as raw_schema
    pdf = pdf.reindex(columns=[f.name for f in raw_schema.fields])  # align column order
    return spark.createDataFrame(pdf, schema=raw_schema)


def read_any(path: str) -> DataFrame:
    """Dispatch to the correct reader based on file extension."""
    if path.lower().endswith(".csv"):
        return read_csv_file(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        return read_excel_file(path)
    else:
        raise ValueError(f"Unsupported file format for: {path}")


def load_daily_batch(paths: list) -> DataFrame:
    """Read and union all files (csv and/or xlsx) that landed for the current batch."""
    dfs = [read_any(p) for p in paths]
    df = dfs[0]
    for other in dfs[1:]:
        df = df.unionByName(other)
    return df


raw_df = load_daily_batch(source_files)
print(f"Loaded {raw_df.count()} raw rows from {len(source_files)} file(s)")
raw_df.show(5, truncate=False)


StatementMeta(kmspark, 2, 5, Finished, Available, Finished, False)

Loaded 1000000 raw rows from 1 file(s)
+--------------+-------------------+-----------------+-------------------------------------------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------------+
|Transaction_ID|Date               |Customer_Name    |Product                                                |Total_Items|Total_Cost|Payment_Method|City         |Store_Type      |Discount_Applied|Customer_Category|Season|Promotion                 |
+--------------+-------------------+-----------------+-------------------------------------------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------------+
|1000000000    |2022-01-21 06:27:29|Stacey Price     |['Ketchup', 'Shaving Cream', 'Light Bulbs']            |3          |71.65     |Mobile Payment|Los Angeles  |Warehouse Club  |True            |Homemaker        |

In [5]:
bronze_df = (
    raw_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_batch_date", F.lit(batch_date))
)

print("Bronze schema:")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)


StatementMeta(kmspark, 2, 6, Finished, Available, Finished, False)

Bronze schema:
root
 |-- Transaction_ID: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: string (nullable = true)
 |-- Total_Cost: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: string (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_batch_date: string (nullable = false)

+--------------+-------------------+-----------------+-------------------------------------------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------------+--------------------------+-----------------+
|Transaction_ID|Date         

In [6]:
text_columns = [
    "Customer_Name", "Payment_Method", "City", "Store_Type",
    "Customer_Category", "Season", "Promotion"
]

silver_df = bronze_df

# Trim whitespace on every string column, Bronze's audit columns included
for col_name in bronze_df.columns:
    if dict(bronze_df.dtypes)[col_name] == "string":
        silver_df = silver_df.withColumn(col_name, F.trim(F.col(col_name)))

# Standardize categorical/text columns to a consistent case
# (Title Case for names/cities, as it reads best for reporting)
for col_name in ["Customer_Name", "City", "Store_Type", "Customer_Category", "Season", "Promotion", "Payment_Method"]:
    silver_df = silver_df.withColumn(col_name, F.initcap(F.col(col_name)))

# Normalize obviously malformed / placeholder values to real nulls
malformed_tokens = ["", "NA", "N/A", "NULL", "None", "-"]
for col_name in text_columns:
    silver_df = silver_df.withColumn(
        col_name,
        F.when(F.col(col_name).isin(malformed_tokens), None).otherwise(F.col(col_name))
    )


StatementMeta(kmspark, 2, 7, Finished, Available, Finished, False)

In [7]:
silver_df = (
    silver_df
    # Date -> timestamp: نجرب صيغة فيها وقت، وبعدين صيغ التاريخ بس كـ احتياط
    .withColumn(
        "Date",
        F.coalesce(
            F.to_timestamp(F.col("Date"), "yyyy-MM-dd HH:mm:ss"),   # ⭐ الصيغة الحقيقية بتاعتك
            F.to_timestamp(F.col("Date"), "yyyy-MM-dd"),
            F.to_timestamp(F.col("Date"), "dd-MM-yyyy HH:mm:ss"),
            F.to_timestamp(F.col("Date"), "dd/MM/yyyy HH:mm:ss"),
            F.to_timestamp(F.col("Date"), "MM/dd/yyyy HH:mm:ss"),
        )
    )
    # Total_Items -> integer
    .withColumn("Total_Items", F.col("Total_Items").cast(IntegerType()))
    # Total_Cost -> double
    .withColumn("Total_Cost", F.col("Total_Cost").cast(DoubleType()))
    # Discount_Applied -> boolean (handles Yes/No, True/False, 1/0 style source values)
    .withColumn(
        "Discount_Applied",
        F.when(F.lower(F.col("Discount_Applied")).isin("yes", "true", "1"), True)
         .when(F.lower(F.col("Discount_Applied")).isin("no", "false", "0"), False)
         .otherwise(None)
         .cast(BooleanType())
    )
)

StatementMeta(kmspark, 2, 8, Finished, Available, Finished, False)

In [8]:
silver_df = (
    silver_df
    .withColumn("Year", F.year("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("Day", F.dayofmonth("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("Month_Name", F.date_format("Date", "MMMM"))
)


StatementMeta(kmspark, 2, 9, Finished, Available, Finished, False)

In [9]:
silver_df = silver_df.withColumn(
    "Product_Array",
    F.transform(F.split(F.col("Product"), ","), lambda x: F.trim(x))
)


StatementMeta(kmspark, 2, 10, Finished, Available, Finished, False)

In [10]:
silver_df_validated = silver_df.filter(
    F.col("Transaction_ID").isNotNull()
    & F.col("Date").isNotNull()
    & F.col("Total_Items").isNotNull() & (F.col("Total_Items") > 0)
    & F.col("Total_Cost").isNotNull() & (F.col("Total_Cost") >= 0)
    & (F.size(F.col("Product_Array")) > 0)
)

dropped = silver_df.count() - silver_df_validated.count()
print(f"Silver validation dropped {dropped} invalid row(s)")

silver_df_final = silver_df_validated.dropDuplicates(["Transaction_ID"])

print("Silver schema:")
silver_df_final.printSchema()
silver_df_final.show(5, truncate=False)


StatementMeta(kmspark, 2, 11, Finished, Available, Finished, False)

Silver validation dropped 0 invalid row(s)
Silver schema:
root
 |-- Transaction_ID: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: boolean (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_batch_date: string (nullable = false)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month_Name: string (nullable = true)
 |-- Product_Array: array (nullable = true)
 |    |-- element: string (containsNull

In [11]:
customer_window = Window.orderBy("Customer_Name", "Customer_Category")

dim_customer = (
    silver_df_final
    .select("Customer_Name", "Customer_Category")
    .distinct()
    .withColumn("Customer_Key", F.row_number().over(customer_window))
    .select("Customer_Key", "Customer_Name", "Customer_Category")
)

dim_customer.show(5, truncate=False)


StatementMeta(kmspark, 2, 12, Finished, Available, Finished, False)

+------------+-------------+-----------------+
|Customer_Key|Customer_Name|Customer_Category|
+------------+-------------+-----------------+
|1           |Aaron Acevedo|Professional     |
|2           |Aaron Acevedo|Senior Citizen   |
|3           |Aaron Acevedo|Teenager         |
|4           |Aaron Acosta |Homemaker        |
|5           |Aaron Acosta |Middle-aged      |
+------------+-------------+-----------------+
only showing top 5 rows



In [12]:
product_window = Window.orderBy("Product_Name")

dim_product = (
    silver_df_final
    .select(F.explode("Product_Array").alias("Product_Name"))
    .distinct()
    .withColumn("Product_Key", F.row_number().over(product_window))
    .select("Product_Key", "Product_Name")
)

dim_product.show(5, truncate=False)


StatementMeta(kmspark, 2, 13, Finished, Available, Finished, False)

+-----------+----------------+
|Product_Key|Product_Name    |
+-----------+----------------+
|1          |'Air Freshener' |
|2          |'Air Freshener']|
|3          |'Apple'         |
|4          |'Apple']        |
|5          |'BBQ Sauce'     |
+-----------+----------------+
only showing top 5 rows



In [13]:
date_window = Window.orderBy("Full_Date")

dim_date = (
    silver_df_final
    .select(
        F.col("Date").alias("Full_Date"),
        "Day", "Month", "Month_Name", "Quarter", "Year", "Season"
    )
    .distinct()
    .withColumn("Date_Key", F.row_number().over(date_window))
    .select("Date_Key", "Full_Date", "Day", "Month", "Month_Name", "Quarter", "Year", "Season")
)

dim_date.show(5, truncate=False)


StatementMeta(kmspark, 2, 14, Finished, Available, Finished, False)

+--------+-------------------+---+-----+----------+-------+----+------+
|Date_Key|Full_Date          |Day|Month|Month_Name|Quarter|Year|Season|
+--------+-------------------+---+-----+----------+-------+----+------+
|1       |2020-01-01 00:03:54|1  |1    |January   |1      |2020|Fall  |
|2       |2020-01-01 00:07:48|1  |1    |January   |1      |2020|Spring|
|3       |2020-01-01 00:08:42|1  |1    |January   |1      |2020|Winter|
|4       |2020-01-01 00:09:02|1  |1    |January   |1      |2020|Summer|
|5       |2020-01-01 00:10:25|1  |1    |January   |1      |2020|Winter|
+--------+-------------------+---+-----+----------+-------+----+------+
only showing top 5 rows



In [14]:
store_window = Window.orderBy("City", "Store_Type")

dim_store = (
    silver_df_final
    .select("City", "Store_Type")
    .distinct()
    .withColumn("Store_Key", F.row_number().over(store_window))
    .select("Store_Key", "City", "Store_Type")
)

dim_store.show(5, truncate=False)


StatementMeta(kmspark, 2, 15, Finished, Available, Finished, False)

+---------+-------+-----------------+
|Store_Key|City   |Store_Type       |
+---------+-------+-----------------+
|1        |Atlanta|Convenience Store|
|2        |Atlanta|Department Store |
|3        |Atlanta|Pharmacy         |
|4        |Atlanta|Specialty Store  |
|5        |Atlanta|Supermarket      |
+---------+-------+-----------------+
only showing top 5 rows



In [15]:
promotion_window = Window.orderBy("Promotion", "Discount_Applied")

dim_promotion = (
    silver_df_final
    .select("Promotion", "Discount_Applied")
    .distinct()
    .withColumn("Promotion_Key", F.row_number().over(promotion_window))
    .select("Promotion_Key", "Promotion", "Discount_Applied")
)

dim_promotion.show(5, truncate=False)


StatementMeta(kmspark, 2, 16, Finished, Available, Finished, False)

+-------------+--------------------------+----------------+
|Promotion_Key|Promotion                 |Discount_Applied|
+-------------+--------------------------+----------------+
|1            |NULL                      |false           |
|2            |NULL                      |true            |
|3            |Bogo (buy One Get One)    |false           |
|4            |Bogo (buy One Get One)    |true            |
|5            |Discount On Selected Items|false           |
+-------------+--------------------------+----------------+
only showing top 5 rows



In [16]:
fact_sales = (
    silver_df_final.alias("s")
    .join(dim_customer.alias("c"), on=["Customer_Name", "Customer_Category"], how="left")
    .join(dim_store.alias("st"), on=["City", "Store_Type"], how="left")
    .join(dim_promotion.alias("p"), on=["Promotion", "Discount_Applied"], how="left")
    .join(dim_date.alias("d"), F.col("s.Date") == F.col("d.Full_Date"), how="left")
    .select(
        F.col("s.Transaction_ID"),
        F.col("d.Date_Key"),
        F.col("c.Customer_Key"),
        F.col("st.Store_Key"),
        F.col("p.Promotion_Key"),
        F.col("s.Total_Items"),
        F.col("s.Total_Cost"),
    )
)

fact_sales.show(5, truncate=False)


StatementMeta(kmspark, 2, 17, Finished, Available, Finished, False)

+--------------+--------+------------+---------+-------------+-----------+----------+
|Transaction_ID|Date_Key|Customer_Key|Store_Key|Promotion_Key|Total_Items|Total_Cost|
+--------------+--------+------------+---------+-------------+-----------+----------+
|1000000013    |741571  |18229       |60       |3            |7          |17.51     |
|1000000026    |165122  |125562      |24       |5            |6          |49.45     |
|1000000011    |739930  |294783      |46       |5            |7          |31.88     |
|1000000025    |27841   |301542      |43       |3            |10         |7.82      |
|1000000027    |53714   |393746      |10       |NULL         |10         |29.36     |
+--------------+--------+------------+---------+-------------+-----------+----------+
only showing top 5 rows



In [17]:
fact_product_bridge = (
    silver_df_final
    .select("Transaction_ID", F.explode("Product_Array").alias("Product_Name"))
    .join(dim_product, on="Product_Name", how="left")
    .select("Transaction_ID", "Product_Key")
)

fact_product_bridge.show(5, truncate=False)


StatementMeta(kmspark, 2, 18, Finished, Available, Finished, False)

+--------------+-----------+
|Transaction_ID|Product_Key|
+--------------+-----------+
|1000000008    |230        |
|1000000013    |311        |
|1000000013    |3          |
|1000000013    |87         |
|1000000013    |60         |
+--------------+-----------+
only showing top 5 rows



In [18]:
def write_csv(df: DataFrame, path: str, partition_col: str = None):
    """Write a DataFrame as CSV with header, overwriting any existing data at the path."""
    writer = df.coalesce(1).write.mode("overwrite").option("header", True)
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.csv(path)


# ---- Bronze ----
write_csv(bronze_df, f"{bronze_path}/batch_date={batch_date}")

# ---- Silver ----
# نحول Product_Array لنص مفصول بفاصلة عشان CSV يقدر يستوعبه
silver_df_to_write = silver_df_final.withColumn(
    "Product_Array", F.concat_ws(", ", F.col("Product_Array"))
)
write_csv(silver_df_to_write, silver_path)

# ---- Gold: dimensions + fact + bridge ----
write_csv(dim_customer,        f"{gold_path}/DimCustomer")
write_csv(dim_product,         f"{gold_path}/DimProduct")
write_csv(dim_date,            f"{gold_path}/DimDate")
write_csv(dim_store,           f"{gold_path}/DimStore")
write_csv(dim_promotion,       f"{gold_path}/DimPromotion")
write_csv(fact_sales,          f"{gold_path}/FactSales")
write_csv(fact_product_bridge, f"{gold_path}/FactProductBridge")

print("Bronze, Silver, and Gold layers written successfully.")
print(f"Bronze -> {bronze_path}/batch_date={batch_date}")
print(f"Silver -> {silver_path}")
print(f"Gold   -> {gold_path}")

StatementMeta(kmspark, 2, 19, Finished, Available, Finished, False)

Bronze, Silver, and Gold layers written successfully.
Bronze -> abfss://retail@knoledgers.dfs.core.windows.net/bronze/transactions/batch_date=2026-07-07
Silver -> abfss://retail@knoledgers.dfs.core.windows.net/silver/transactions
Gold   -> abfss://retail@knoledgers.dfs.core.windows.net/gold


In [19]:
print("=== Row Counts ===")
print(f"Bronze         : {bronze_df.count()}")
print(f"Silver         : {silver_df_final.count()}")
print(f"DimCustomer    : {dim_customer.count()}")
print(f"DimProduct     : {dim_product.count()}")
print(f"DimDate        : {dim_date.count()}")
print(f"DimStore       : {dim_store.count()}")
print(f"DimPromotion   : {dim_promotion.count()}")
print(f"FactSales      : {fact_sales.count()}")
print(f"FactProductBridge : {fact_product_bridge.count()}")


StatementMeta(kmspark, 2, 20, Finished, Available, Finished, False)

=== Row Counts ===
Bronze         : 1000000
Silver         : 1000000
DimCustomer    : 676645
DimProduct     : 324
DimDate        : 999093
DimStore       : 60
DimPromotion   : 6
FactSales      : 1005523
FactProductBridge : 3000343


In [20]:
# شكل الداتا وعدد الصفوف/الأعمدة
print(f"عدد الصفوف: {silver_df_final.count()}")
print(f"عدد الأعمدة: {len(silver_df_final.columns)}")
print("\n=== أسماء الأعمدة وأنواعها ===")
silver_df_final.printSchema()

# عرض عينة
silver_df_final.show(5, truncate=False)

StatementMeta(kmspark, 2, 21, Finished, Available, Finished, False)

عدد الصفوف: 1000000
عدد الأعمدة: 21

=== أسماء الأعمدة وأنواعها ===
root
 |-- Transaction_ID: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: boolean (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_batch_date: string (nullable = false)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month_Name: string (nullable = true)
 |-- Product_Array: array (nullable = true)
 |    |-- element: string (co

In [21]:
# يديك min, max, mean, stddev لكل عمود رقمي
silver_df_final.select("Total_Items", "Total_Cost").describe().show()

# لو عايز أرقام أدق (median, percentiles)
silver_df_final.select("Total_Items", "Total_Cost").summary(
    "count", "min", "25%", "50%", "75%", "max", "mean", "stddev"
).show()

StatementMeta(kmspark, 2, 22, Finished, Available, Finished, False)

+-------+-----------------+------------------+
|summary|      Total_Items|        Total_Cost|
+-------+-----------------+------------------+
|  count|          1000000|           1000000|
|   mean|         5.495941| 52.45522039999992|
| stddev|2.871654187209303|27.416989145333176|
|    min|                1|               5.0|
|    max|               10|             100.0|
+-------+-----------------+------------------+

+-------+------------------+------------------+
|summary|       Total_Items|        Total_Cost|
+-------+------------------+------------------+
|  count|           1000000|           1000000|
|    min|                 1|               5.0|
|    25%|                 3|             28.71|
|    50%|                 5|             52.41|
|    75%|                 8|             76.18|
|    max|                10|             100.0|
|   mean|          5.495941|52.455220399999924|
| stddev|2.8716541872093027|27.416989145333176|
+-------+------------------+------------------+


In [22]:
from pyspark.sql import functions as F

null_counts = silver_df_final.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_df_final.columns
])
null_counts.show(vertical=True, truncate=False)

StatementMeta(kmspark, 2, 23, Finished, Available, Finished, False)

-RECORD 0---------------------
 Transaction_ID      | 0      
 Date                | 0      
 Customer_Name       | 0      
 Product             | 0      
 Total_Items         | 0      
 Total_Cost          | 0      
 Payment_Method      | 0      
 City                | 0      
 Store_Type          | 0      
 Discount_Applied    | 0      
 Customer_Category   | 0      
 Season              | 0      
 Promotion           | 333943 
 ingestion_timestamp | 0      
 source_batch_date   | 0      
 Year                | 0      
 Month               | 0      
 Day                 | 0      
 Quarter             | 0      
 Month_Name          | 0      
 Product_Array       | 0      



In [23]:
categorical_cols = ["Customer_Category", "City", "Store_Type", 
                     "Payment_Method", "Season", "Promotion"]

for col_name in categorical_cols:
    distinct_count = silver_df_final.select(col_name).distinct().count()
    print(f"{col_name}: {distinct_count} قيمة مميزة")

StatementMeta(kmspark, 2, 24, Finished, Available, Finished, False)

Customer_Category: 8 قيمة مميزة
City: 10 قيمة مميزة
Store_Type: 6 قيمة مميزة
Payment_Method: 4 قيمة مميزة
Season: 4 قيمة مميزة
Promotion: 3 قيمة مميزة


In [24]:
for col_name in categorical_cols:
    print(f"\n=== توزيع {col_name} ===")
    silver_df_final.groupBy(col_name).count().orderBy(F.desc("count")).show(10, truncate=False)

StatementMeta(kmspark, 2, 25, Finished, Available, Finished, False)


=== توزيع Customer_Category ===
+-----------------+------+
|Customer_Category|count |
+-----------------+------+
|Senior Citizen   |125485|
|Homemaker        |125418|
|Teenager         |125319|
|Retiree          |125072|
|Student          |124842|
|Professional     |124651|
|Middle-aged      |124636|
|Young Adult      |124577|
+-----------------+------+


=== توزيع City ===
+-------------+------+
|City         |count |
+-------------+------+
|Boston       |100566|
|Dallas       |100559|
|Seattle      |100167|
|Chicago      |100059|
|Houston      |100050|
|New York     |100007|
|Los Angeles  |99879 |
|Miami        |99839 |
|San Francisco|99808 |
|Atlanta      |99066 |
+-------------+------+


=== توزيع Store_Type ===
+-----------------+------+
|Store_Type       |count |
+-----------------+------+
|Supermarket      |166936|
|Pharmacy         |166915|
|Convenience Store|166749|
|Warehouse Club   |166685|
|Department Store |166614|
|Specialty Store  |166101|
+-----------------+------+


=

In [25]:
silver_df_final.select(
    F.min("Date").alias("أقدم تاريخ"),
    F.max("Date").alias("أحدث تاريخ")
).show(truncate=False)

StatementMeta(kmspark, 2, 26, Finished, Available, Finished, False)

+-------------------+-------------------+
|أقدم تاريخ         |أحدث تاريخ         |
+-------------------+-------------------+
|2020-01-01 00:03:54|2024-05-18 19:31:03|
+-------------------+-------------------+



In [26]:
# كام منتج في المتوسط لكل عملية بيع
silver_df_final.select(F.size("Product_Array").alias("num_products")).describe().show()

# أشهر المنتجات
silver_df_final.select(F.explode("Product_Array").alias("Product_Name")) \
    .groupBy("Product_Name").count().orderBy(F.desc("count")).show(20, truncate=False)

# عدد المنتجات الفريدة الكلي
distinct_products = silver_df_final.select(F.explode("Product_Array").alias("p")).distinct().count()
print(f"عدد المنتجات الفريدة: {distinct_products}")

StatementMeta(kmspark, 2, 27, Finished, Available, Finished, False)

+-------+------------------+
|summary|      num_products|
+-------+------------------+
|  count|           1000000|
|   mean|          3.000343|
| stddev|1.4139380755789424|
|    min|                 1|
|    max|                 5|
+-------+------------------+

+-------------------+-----+
|Product_Name       |count|
+-------------------+-----+
|'Toothpaste'       |29399|
|'Toothpaste']      |19525|
|['Toothpaste'      |19507|
|'Chips'            |14898|
|'Ironing Board'    |14892|
|'Cleaning Spray'   |14880|
|'Yogurt'           |14874|
|'Orange'           |14866|
|'Olive Oil'        |14834|
|'Deodorant'        |14831|
|'Mayonnaise'       |14828|
|'Jam'              |14810|
|'Toothbrush'       |14768|
|'Ice Cream'        |14767|
|'Bath Towels'      |14754|
|'Peanut Butter'    |14751|
|'Razors'           |14750|
|'Soda'             |14747|
|'Laundry Detergent'|14738|
|'Air Freshener'    |14737|
+-------------------+-----+
only showing top 20 rows

عدد المنتجات الفريدة: 324


In [27]:
# متوسط قيمة البيع حسب الموسم
silver_df_final.groupBy("Season").agg(
    F.count("*").alias("عدد العمليات"),
    F.avg("Total_Cost").alias("متوسط قيمة البيع"),
    F.sum("Total_Cost").alias("إجمالي المبيعات")
).orderBy(F.desc("إجمالي المبيعات")).show()

# متوسط قيمة البيع حسب المدينة
silver_df_final.groupBy("City").agg(
    F.count("*").alias("عدد العمليات"),
    F.avg("Total_Cost").alias("متوسط قيمة البيع")
).orderBy(F.desc("عدد العمليات")).show(10)

StatementMeta(kmspark, 2, 28, Finished, Available, Finished, False)

+------+------------+------------------+--------------------+
|Season|عدد العمليات|  متوسط قيمة البيع|     إجمالي المبيعات|
+------+------------+------------------+--------------------+
|  Fall|      250248| 52.49557922540836|1.3136913709999992E7|
|Summer|      249621| 52.54636344698546|1.3116675789999958E7|
|Spring|      250368| 52.37585773741056|1.3113238750000007E7|
|Winter|      249763|52.403246878040434|1.3088392150000013E7|
+------+------------+------------------+--------------------+

+-------------+------------+------------------+
|         City|عدد العمليات|  متوسط قيمة البيع|
+-------------+------------+------------------+
|       Boston|      100566| 52.33685301195246|
|       Dallas|      100559| 52.47776459590879|
|      Seattle|      100167| 52.26636946299673|
|      Chicago|      100059|  52.6008400043974|
|      Houston|      100050| 52.44432563718133|
|     New York|      100007| 52.52102272840906|
|  Los Angeles|       99879|52.387320557875135|
|        Miami|       9

In [28]:
# أعلى وأقل قيم Total_Cost
silver_df_final.orderBy(F.desc("Total_Cost")).select(
    "Transaction_ID", "Total_Cost", "Total_Items"
).show(10)

silver_df_final.orderBy(F.asc("Total_Cost")).select(
    "Transaction_ID", "Total_Cost", "Total_Items"
).show(10)

StatementMeta(kmspark, 2, 29, Finished, Available, Finished, False)

+--------------+----------+-----------+
|Transaction_ID|Total_Cost|Total_Items|
+--------------+----------+-----------+
|    1000021018|     100.0|          1|
|    1000830259|     100.0|          2|
|    1000509448|     100.0|          6|
|    1000372516|     100.0|          7|
|    1000515730|     100.0|          8|
|    1000144459|     100.0|          5|
|    1000714488|     100.0|          5|
|    1000591212|     100.0|          1|
|    1000140221|     100.0|          4|
|    1000008258|     100.0|          5|
+--------------+----------+-----------+
only showing top 10 rows

+--------------+----------+-----------+
|Transaction_ID|Total_Cost|Total_Items|
+--------------+----------+-----------+
|    1000411429|       5.0|          1|
|    1000895136|       5.0|          5|
|    1000423271|       5.0|          9|
|    1000549537|       5.0|          2|
|    1000199054|       5.0|          9|
|    1000498943|       5.0|          8|
|    1000379793|       5.0|          3|
|    100056820